### HyDE
🧠 What is HyDe?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user’s query directly, you first generate a hypothetical answer (document) to the query using an LLM — and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short
2. Language mismatch between query and documents
3.You want to retrieve based on answer content, not question words

In [3]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


In [4]:
# 1. Load and chunk your dataset
chunk_size = 300
chunk_overlap = 100

# loading data
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()

# text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
docs = text_splitter.split_documents(documents=documents)
docs

[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and s

In [6]:
# 2. Build vector store
from langchain_community.vectorstores import FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1871.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# 3. Set up the LLM you’ll use to generate hypothetical answers
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm = init_chat_model("groq:llama-3.1-8b-instant")

In [10]:
## creating vector store
db = Chroma.from_documents(documents = docs,embedding=embeddings,persist_directory = "output/steve_jobs_for_hyde.db")
##create the retriever
base_retriever=db.as_retriever(search_kwargs = {"k":5})

In [12]:
from langchain_core.output_parsers import StrOutputParser
## Generating a prompt gor generating HyDE
from langchain_core.prompts import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template = template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query = query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc

In [13]:
query = 'When was Steve Jobs fired from Apple?'
print(get_hyde_doc(query=query))

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
**The Firing of Steve Jobs from Apple: A Turning Point in History**

Steve Jobs, the visionary co-founder of Apple, was a pivotal figure in the development of the company into the technological giant it is today. However, in a shocking move, Jobs was eventually ousted from the company he helped create. This pivotal moment in Apple's history occurred on September 29, 1985, when the Apple Board of Directors, led by then-Chairman John Sculley, voted to remove Steve Jobs from his position as CEO.

**The Circumstances Leading to the Firing**

During the early 1980s, Apple experienced explosive growth, and the company's stock price soared. However, as the company expanded, tensions began to rise between Steve Jobs and John Sculley, who was hired as CEO in 1983. Scul

In [14]:
matched_doc = base_retriever.invoke(get_hyde_doc(query))
print(matched_doc)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Steve_Jobs', 'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was bor

### Langchain-HypotheticalDocumentEmbedder

In [4]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Step 1: LLM
llm = init_chat_model("groq:llama-3.1-8b-instant")

# Step 2: Load and split documents
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50).split_documents(docs)

# Step 3: Base embeddings and vector store
base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=base_embeddings,
    persist_directory="output/langchain"
)

# Step 4: HyDE chain — generate hypothetical answer, embed it, then retrieve
hyde_chain = (
    PromptTemplate.from_template("Generate a concise hypothetical answer for this topic: {query}")
    | llm
    | StrOutputParser()
)

# Step 5: RAG answering chain
rag_chain = (
    PromptTemplate.from_template("""Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
    | llm
    | StrOutputParser()
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5671.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:

# Step 6: Full HyDE RAG pipeline
def hyde_rag_pipeline(query):
    hypothetical_doc = hyde_chain.invoke({"query": query})
    print(f"🧠 Hypothetical doc:\n{hypothetical_doc}\n")
    context = "\n\n".join(
        doc.page_content for doc in vectorstore.similarity_search(hypothetical_doc, k=4)
    )
    return rag_chain.invoke({"input": query, "context": context})


In [6]:

# Step 7: Run example query
answer = hyde_rag_pipeline("What memory modules does LangChain provide?")
print("✅ Final Answer:\n", answer)

🧠 Hypothetical doc:
LangChain is an open-source suite for building conversational AI applications.  As I do not have information about the specific memory modules of LangChain, I'll provide a hypothetical answer based on common memory modules used in conversational AI frameworks.

LangChain provides the following memory modules:

- **LLaMA Memory**: A memory module that utilizes the LLaMA large language model to store and retrieve information.
- **Bert Memory**: A memory module that leverages the pre-trained BERT model to store and access knowledge.
- **SQL Memory**: A memory module that allows users to store and retrieve structured data from SQL databases.
- **External Memory**: A memory module that enables users to integrate external knowledge graphs or databases into their conversational AI applications.

Please note that this is a hypothetical answer, and actual memory modules provided by LangChain may differ.

✅ Final Answer:
 LangChain provides two memory modules: 

1. Conversati